# Árvore de Decisão — espaço de busca, pipeline e explicabilidade

Dupla Modelos Interpretáveis (Gabriel e Luiza), ART.7, Sprint 04. Este notebook roda o segundo modelo interpretável do grupo: a Regressão Logística (Gabriel, `notebooks/regressao_logistica.ipynb`) traç explicabilidade por odds ratio; esta árvore traz explicabilidade por regras legíveis (cards #233/#234), a segunda via intrínseca que o barema do ART.7 pede.

**Antes de rodar:** monte o Drive, confirme `scikit-learn==1.9.1` e `scipy==1.18.1` (mesma exigência do notebook da Regressão Logística — minor diferente muda resultado de `DecisionTreeClassifier` em caso de empate de split).

In [ ]:
!pip install -q -r requirements.txt
import sklearn, scipy
print('scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)

## 1. Carga da base e matriz do contrato

Reaproveita `matriz.preparar_matriz`, o mesmo contrato que a Regressão Logística usa — é o que garante que as duas subseções da 4.4 comparam os modelos sobre os mesmos dados (card #227).

In [ ]:
# No Colab, monte o Drive antes de rodar esta celula e ajuste o caminho abaixo.
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, 'scripts')

import pandas as pd
from matriz import preparar_matriz

CAMINHO_BASE = 'CAMINHO/PARA/base_analitica.parquet'  # TODO: ajustar
CORTE_VALIDACAO = '2025-07-01'
CORTE_TESTE = '2026-01-01'

df = pd.read_parquet(CAMINHO_BASE)
preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)
from matriz import resumo_da_matriz
resumo_da_matriz(preparo)

## 2. Espaço de busca da Árvore de Decisão (card #228)

Grade declarada em `src/espaco_busca_arvore.py` — ver o módulo para a justificativa completa de cada eixo. Resumo: `max_depth` limitado a 6 e `min_samples_leaf` a partir de 50 porque a árvore final precisa caber numa leitura de regras (cards #233/#234), não só generalizar bem.

In [ ]:
from espaco_busca_arvore import (GRADE_ARVORE, combinacoes, custo_estimado,
                                  medir_ajuste, grade_com_prefixo)

print(f'{len(combinacoes())} combinacoes na grade')
GRADE_ARVORE

### 2.1. Medir o tempo de um ajuste

**Pendência real:** `SEGUNDOS_POR_AJUSTE` em `espaco_busca_arvore.py` está `None` de propósito — ninguém mediu ainda. Rode a célula abaixo sobre a matriz real (mesmo procedimento do card #206 do Gabriel) e cole o valor medido de volta no módulo antes de seguir para a seção 4.

In [ ]:
x_treino = preparo['matrizes']['treino']
y_treino = preparo['y']['treino']

medicao = medir_ajuste(x_treino, y_treino,
                        {'criterion': 'gini', 'max_depth': 6,
                         'min_samples_leaf': 50, 'class_weight': 'balanced'})
medicao  # TODO: copiar 'segundos' para SEGUNDOS_POR_AJUSTE em espaco_busca_arvore.py

## 3. Pipeline sobre o contrato (card #229)

`criar_pipeline` clona o `ColumnTransformer` do contrato — o ajuste real acontece dentro de cada fold da busca, nunca aqui. Esta célula só mede a métrica de partida, antes de qualquer busca de hiperparâmetro.

In [ ]:
from pipeline_arvore import criar_pipeline, ajustar_no_treino, metrica_de_partida

pipeline_base = criar_pipeline(preparo['preprocessador'])
relato_partida = ajustar_no_treino(pipeline_base, preparo['x']['treino'], y_treino)
relato_partida

In [ ]:
# TODO: substituir por 'from card_241 import avaliar' quando aquele card estiver em develop.
# metrica_de_partida(pipeline_base, preparo['x']['treino'], y_treino, avaliar)

## 4. GridSearchCV (card #231)

Usa a grade do card #228 (prefixada pelo nome do passo do pipeline) e os folds agrupados por Cliente de `validacao.criar_folds` — nunca uma partição própria. `scoring='average_precision'` segue a mesma escolha de métrica principal de `modelo.py` (Precisão Média), enquanto o scorer oficial do card #241 não estiver em `develop` — troque por ele quando existir, para as quatro duplas comparar pelo mesmo número.

In [ ]:
from sklearn.model_selection import GridSearchCV
from validacao import criar_folds

folds = criar_folds(preparo['x']['treino'], preparo['grupos']['treino'])
grade_prefixada = grade_com_prefixo('modelo')

busca = GridSearchCV(
    criar_pipeline(preparo['preprocessador']),
    grade_prefixada,
    scoring='average_precision',  # TODO: trocar pelo scorer oficial do card #241
    cv=folds,
    refit=True,
    n_jobs=-1,
)
busca.fit(preparo['x']['treino'], y_treino)

print('best_params_:', busca.best_params_)
print('best_score_:', busca.best_score_)

## 5. Hiperparâmetros vencedores (card #232)

Registra `best_params_` e a tabela completa da busca — a decisão precisa ser reconstruível sem reexecutar a grade inteira.

In [ ]:
import json
from pathlib import Path

resultados = pd.DataFrame(busca.cv_results_)
colunas_uteis = [c for c in resultados.columns
                 if c.startswith('param_') or c in ('mean_test_score', 'std_test_score', 'rank_test_score')]
tabela_busca = resultados[colunas_uteis].sort_values('rank_test_score')
tabela_busca.head(10)

In [ ]:
Path('src/hiperparametros_arvore.json').write_text(
    json.dumps(busca.best_params_, indent=2, ensure_ascii=False), encoding='utf-8'
)
busca.best_params_

## 6. Árvore visual e regras (card #233)

Reconstrói a árvore com os hiperparâmetros vencedores e extrai as regras — esta é a via de explicabilidade intrínseca que o barema pede (ao lado dos odds ratio da Regressão Logística). Mapeie os nomes de saída do `ColumnTransformer` de volta para a feature original antes de interpretar.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
import matplotlib.pyplot as plt

modelo_final = criar_pipeline(preparo['preprocessador'], **busca.best_params_.copy())
# TODO: remover o prefixo 'modelo__' de best_params_ antes de passar aqui,
# ou reaproveitar busca.best_estimator_ diretamente.
modelo_final = busca.best_estimator_

arvore = modelo_final.named_steps['modelo']
nomes_features = modelo_final.named_steps['preparo'].get_feature_names_out()

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(arvore, feature_names=nomes_features, class_names=['nao Detrator', 'Detrator'],
          filled=True, rounded=True, fontsize=8, ax=ax)
plt.show()

In [ ]:
regras = export_text(arvore, feature_names=list(nomes_features))
print(regras)

## 7. Interpretação para a operação (card #234)

Traduzir as regras acima para a linguagem da operação da Azul: condição → previsão → confiança (proporção de Detrator na folha), sempre separando associação de causa e registrando as limitações do modelo (ver `.claude`/card #234 para o roteiro completo).

In [ ]:
# TODO: escrever a interpretacao em markdown depois de ler as regras da secao 6.